# BigQuery: constraints + validation (operational schema)

## What BigQuery can and cannot do

- **Primary keys and foreign keys** in BigQuery are **`NOT ENFORCED`**. Google documents that the `enforced` column in `INFORMATION_SCHEMA.TABLE_CONSTRAINTS` is always **`NO`**: inserts/updates are **not** rejected when a FK is orphaned or a PK is duplicated. Constraints still help the **optimizer** and document the model.
- **CHECK constraints** are **not** available as table DDL in BigQuery the way they are in PostgreSQL.
- **Practical substitute in-BQ**: run the **`ASSERT`** statements in this notebook (or schedule them) so jobs **fail** when data violates the same rules you would express as FK/CHECK in another RDBMS.

**This notebook only** touches BigQuery (no Cloud Function / React changes).

References: [Primary and foreign keys](https://cloud.google.com/bigquery/docs/primary-foreign-keys), [TABLE_CONSTRAINTS](https://cloud.google.com/bigquery/docs/information-schema-table-constraints), [ASSERT](https://cloud.google.com/bigquery/docs/reference/standard-sql/debugging-statements).

---

## Order of operations

1. Configure project/dataset in the next cell.
2. **Inspect** existing PK/FK metadata.
3. **Optional**: run `ALTER TABLE` blocks only if your tables were created **without** the FK/PK lines from `DATABASE_DESIGN.md` §6 (skip if `schema_migration` already applied full DDL).
4. Run **assertions** — expect no output; any failure raises a query error with the message you passed to `ASSERT`.

In [4]:
import os
from google.cloud import bigquery

# Optional: os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/key.json"

PROJECT_ID = os.getenv("BQ_PROJECT_ID", "mke-trees")
DATASET = os.getenv("BQ_DATASET", "mke_tree_dataset")
LOCATION = os.getenv("BQ_LOCATION", "us-central1")

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)


def fq(table: str) -> str:
    return f"`{PROJECT_ID}.{DATASET}.{table}`"


def run(sql: str):
    job = client.query(sql)
    job.result()
    return job


print(f"project={PROJECT_ID} dataset={DATASET} location={LOCATION}")

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

## 1) Inspect declared PK/FK metadata

Expect `constraint_type` of `PRIMARY KEY` / `FOREIGN KEY` and **`enforced` = NO** (BigQuery default).

In [ ]:
sql = f"""
SELECT
  tc.table_name,
  tc.constraint_name,
  tc.constraint_type,
  tc.enforced
FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.TABLE_CONSTRAINTS` tc
WHERE tc.constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY tc.table_name, tc.constraint_type, tc.constraint_name
"""
rows = list(client.query(sql).result())
for r in rows:
    print(dict(r))
print(f"Total: {len(rows)} constraint rows")

## 2) Optional — add missing **declarative** PK/FK

Run **only** if your tables exist but were created without constraints. If you already used the DDL in `DATABASE_DESIGN.md` §6 / `schema_migration.ipynb`, **skip this section** (duplicates will error).

Uncomment and run the statements you need. Named constraints are required for `FOREIGN KEY` via `ALTER TABLE` in BigQuery.

```sql
-- Example pattern (adjust names if constraints already exist):
-- ALTER TABLE `proj.ds.service_request_assignees`
-- ADD CONSTRAINT fk_sra_service_request
-- FOREIGN KEY (service_request_id) REFERENCES `proj.ds.service_requests`(service_request_id) NOT ENFORCED;
```

In [3]:
# Set True only if you need to apply alters from this notebook (usually False).
RUN_OPTIONAL_ALTER_CONSTRAINTS = False

OPTIONAL_ALTER_SQL = f"""
-- Uncomment lines below when tables lack constraints. Run one at a time if errors occur.

-- ALTER TABLE {fq('quarter_sections')}
-- ADD PRIMARY KEY (qs_id) NOT ENFORCED;

-- ALTER TABLE {fq('species')}
-- ADD PRIMARY KEY (species_id) NOT ENFORCED;

-- ALTER TABLE {fq('qs_priority')}
-- ADD PRIMARY KEY (qs_id) NOT ENFORCED;
-- ALTER TABLE {fq('qs_priority')}
-- ADD CONSTRAINT fk_qs_priority_qs FOREIGN KEY (qs_id) REFERENCES {fq('quarter_sections')}(qs_id) NOT ENFORCED;

-- ALTER TABLE {fq('trees_core')}
-- ADD PRIMARY KEY (tree_id) NOT ENFORCED;
-- ALTER TABLE {fq('trees_core')}
-- ADD CONSTRAINT fk_trees_qs FOREIGN KEY (qs_id) REFERENCES {fq('quarter_sections')}(qs_id) NOT ENFORCED;
-- ALTER TABLE {fq('trees_core')}
-- ADD CONSTRAINT fk_trees_species FOREIGN KEY (species_id) REFERENCES {fq('species')}(species_id) NOT ENFORCED;

-- ALTER TABLE {fq('trees_features')}
-- ADD PRIMARY KEY (tree_id) NOT ENFORCED;
-- ALTER TABLE {fq('trees_features')}
-- ADD CONSTRAINT fk_features_tree FOREIGN KEY (tree_id) REFERENCES {fq('trees_core')}(tree_id) NOT ENFORCED;

-- ALTER TABLE {fq('users')}
-- ADD PRIMARY KEY (user_id) NOT ENFORCED;

-- ALTER TABLE {fq('service_requests')}
-- ADD PRIMARY KEY (service_request_id) NOT ENFORCED;
-- ALTER TABLE {fq('service_requests')}
-- ADD CONSTRAINT fk_sr_tree FOREIGN KEY (tree_id) REFERENCES {fq('trees_core')}(tree_id) NOT ENFORCED;
-- ALTER TABLE {fq('service_requests')}
-- ADD CONSTRAINT fk_sr_creator FOREIGN KEY (created_by) REFERENCES {fq('users')}(user_id) NOT ENFORCED;

-- ALTER TABLE {fq('service_request_assignees')}
-- ADD PRIMARY KEY (service_request_id, user_id) NOT ENFORCED;
-- ALTER TABLE {fq('service_request_assignees')}
-- ADD CONSTRAINT fk_sra_sr FOREIGN KEY (service_request_id) REFERENCES {fq('service_requests')}(service_request_id) NOT ENFORCED;
-- ALTER TABLE {fq('service_request_assignees')}
-- ADD CONSTRAINT fk_sra_user FOREIGN KEY (user_id) REFERENCES {fq('users')}(user_id) NOT ENFORCED;
-- ALTER TABLE {fq('service_request_assignees')}
-- ADD CONSTRAINT fk_sra_assigned_by FOREIGN KEY (assigned_by) REFERENCES {fq('users')}(user_id) NOT ENFORCED;

-- Optional: only if table exists
-- ALTER TABLE {fq('tree_inspections')}
-- ADD PRIMARY KEY (inspection_id) NOT ENFORCED;
-- ALTER TABLE {fq('tree_inspections')}
-- ADD CONSTRAINT fk_insp_tree FOREIGN KEY (tree_id) REFERENCES {fq('trees_core')}(tree_id) NOT ENFORCED;
-- ALTER TABLE {fq('tree_inspections')}
-- ADD CONSTRAINT fk_insp_inspector FOREIGN KEY (inspector_id) REFERENCES {fq('users')}(user_id) NOT ENFORCED;
"""

if RUN_OPTIONAL_ALTER_CONSTRAINTS:
    raise RuntimeError(
        "Edit OPTIONAL_ALTER_SQL: uncomment needed ALTERs, then run them manually or split into separate run() calls. "
        "BigQuery will error if constraints already exist."
    )
else:
    print("Skipping optional ALTER (set RUN_OPTIONAL_ALTER_CONSTRAINTS = True after editing SQL).")

NameError: name 'fq' is not defined

## 3) Data validation — closest equivalent to **enforced** FK/CHECK

Each `ASSERT` fails the job if the condition is false. Together they cover:

- **PK uniqueness** (no duplicate keys)
- **FK orphans** (child rows without parents)
- **CHECK-style rules** from `DATABASE_DESIGN.md` §9 where applicable (completed_at vs status, lat/lon, dbh, active tree species_id)

If `tree_inspections` does not exist, comment out or remove those asserts.

In [2]:
T = {k: fq(k) for k in [
    'quarter_sections', 'species', 'qs_priority', 'trees_core', 'trees_features',
    'users', 'service_requests', 'service_request_assignees', 'tree_inspections',
]}

ASSERTIONS = [
    # --- PK uniqueness (engine does not enforce) ---
    f"ASSERT NOT EXISTS (SELECT qs_id FROM {T['quarter_sections']} GROUP BY qs_id HAVING COUNT(*) > 1) AS 'quarter_sections: duplicate qs_id';",
    f"ASSERT NOT EXISTS (SELECT species_id FROM {T['species']} GROUP BY species_id HAVING COUNT(*) > 1) AS 'species: duplicate species_id';",
    f"ASSERT NOT EXISTS (SELECT qs_id FROM {T['qs_priority']} GROUP BY qs_id HAVING COUNT(*) > 1) AS 'qs_priority: duplicate qs_id';",
    f"ASSERT NOT EXISTS (SELECT tree_id FROM {T['trees_core']} GROUP BY tree_id HAVING COUNT(*) > 1) AS 'trees_core: duplicate tree_id';",
    f"ASSERT NOT EXISTS (SELECT tree_id FROM {T['trees_features']} GROUP BY tree_id HAVING COUNT(*) > 1) AS 'trees_features: duplicate tree_id';",
    f"ASSERT NOT EXISTS (SELECT user_id FROM {T['users']} GROUP BY user_id HAVING COUNT(*) > 1) AS 'users: duplicate user_id';",
    f"ASSERT NOT EXISTS (SELECT service_request_id FROM {T['service_requests']} GROUP BY service_request_id HAVING COUNT(*) > 1) AS 'service_requests: duplicate id';",
    f"ASSERT NOT EXISTS (SELECT service_request_id, user_id FROM {T['service_request_assignees']} GROUP BY 1,2 HAVING COUNT(*) > 1) AS 'service_request_assignees: duplicate pair';",
    # --- FK-ish orphans ---
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['qs_priority']} p LEFT JOIN {T['quarter_sections']} q USING (qs_id) WHERE q.qs_id IS NULL) AS 'qs_priority: orphaned qs_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} t LEFT JOIN {T['quarter_sections']} q ON t.qs_id = q.qs_id WHERE t.qs_id IS NOT NULL AND q.qs_id IS NULL) AS 'trees_core: orphaned qs_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} t LEFT JOIN {T['species']} s ON t.species_id = s.species_id WHERE t.species_id IS NOT NULL AND s.species_id IS NULL) AS 'trees_core: orphaned species_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_features']} f LEFT JOIN {T['trees_core']} t USING (tree_id) WHERE t.tree_id IS NULL) AS 'trees_features: orphaned tree_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_requests']} r LEFT JOIN {T['trees_core']} t USING (tree_id) WHERE t.tree_id IS NULL) AS 'service_requests: orphaned tree_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_requests']} r LEFT JOIN {T['users']} u ON r.created_by = u.user_id WHERE u.user_id IS NULL) AS 'service_requests: orphaned created_by';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_request_assignees']} a LEFT JOIN {T['service_requests']} r USING (service_request_id) WHERE r.service_request_id IS NULL) AS 'assignees: orphaned service_request_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_request_assignees']} a LEFT JOIN {T['users']} u ON a.user_id = u.user_id WHERE u.user_id IS NULL) AS 'assignees: orphaned user_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_request_assignees']} a LEFT JOIN {T['users']} u ON a.assigned_by = u.user_id WHERE a.assigned_by IS NOT NULL AND u.user_id IS NULL) AS 'assignees: orphaned assigned_by';",
    # --- CHECK-style (§9 subset) ---
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} WHERE dbh IS NOT NULL AND dbh <= 0) AS 'trees_core: dbh must be > 0 when set';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} WHERE height IS NOT NULL AND height < 0) AS 'trees_core: height must be >= 0 when set';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} WHERE latitude IS NOT NULL AND (latitude < -90 OR latitude > 90)) AS 'trees_core: latitude out of range';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} WHERE longitude IS NOT NULL AND (longitude < -180 OR longitude > 180)) AS 'trees_core: longitude out of range';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['trees_core']} WHERE LOWER(status) = 'active' AND species_id IS NULL) AS 'trees_core: active tree requires species_id';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_requests']} WHERE LOWER(status) = 'completed' AND completed_at IS NULL) AS 'service_requests: completed requires completed_at';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['service_requests']} WHERE LOWER(status) != 'completed' AND completed_at IS NOT NULL) AS 'service_requests: completed_at only when completed';",
    f"ASSERT NOT EXISTS (SELECT 1 FROM {T['users']} WHERE role NOT IN ('admin','arborist','viewer')) AS 'users: invalid role';",
]

for i, stmt in enumerate(ASSERTIONS, 1):
    print(f"[{i}/{len(ASSERTIONS)}] running...")
    run(stmt)
print("All assertions passed.")

NameError: name 'fq' is not defined

### 3b) Optional `tree_inspections` asserts

Skip if you dropped this table. Uncomment the loop body after verifying the table exists.

In [ ]:
RUN_TREE_INSPECTION_ASSERTS = False

if RUN_TREE_INSPECTION_ASSERTS:
    insp = fq('tree_inspections')
    tc = fq('trees_core')
    u = fq('users')
    extra = [
        f"ASSERT NOT EXISTS (SELECT inspection_id FROM {insp} GROUP BY inspection_id HAVING COUNT(*) > 1) AS 'tree_inspections: duplicate inspection_id';",
        f"ASSERT NOT EXISTS (SELECT 1 FROM {insp} i LEFT JOIN {tc} t USING (tree_id) WHERE t.tree_id IS NULL) AS 'tree_inspections: orphaned tree_id';",
        f"ASSERT NOT EXISTS (SELECT 1 FROM {insp} i LEFT JOIN {u} x ON i.inspector_id = x.user_id WHERE x.user_id IS NULL) AS 'tree_inspections: orphaned inspector_id';",
        f"ASSERT NOT EXISTS (SELECT 1 FROM {insp} WHERE condition_rating IS NOT NULL AND (condition_rating < 0 OR condition_rating > 5)) AS 'tree_inspections: condition_rating 0..5';",
    ]
    for stmt in extra:
        run(stmt)
    print("tree_inspections assertions passed.")
else:
    print("Skipping tree_inspections (set RUN_TREE_INSPECTION_ASSERTS = True if table exists).")